# 🐦 PSO Feature Selection — Cross-Subject CV
### 3 Dataset × 3 Classifier (SVM · RF · NB)

**Struktur eksperimen:**
- Dataset   : MFCC | LPCC | COMBINED
- Classifier: LinearSVC | RandomForest | GaussianNB
- CV Method : Leave-One-Subject-Out (LOSO / cross-subject)
- Total     : **9 eksperimen** (3 dataset × 3 classifier)
- Parameter PSO: 1 set fixed (tidak ada grid search)


## ✅ Versi Auto Resume / Auto Skip

Notebook ini sudah ditambahkan fitur resume dari `pso_results_live.csv`. Jika eksperimen seperti `MFCC_SVM` sudah selesai dan parameter PSO masih sama, eksperimen tersebut akan otomatis di-skip. Hasil live lama dibaca dulu, lalu hasil baru ditambahkan ke list dan `pso_results_live.csv` disimpan ulang lengkap agar tidak kehilangan hasil sebelumnya.


## 📁 Cell 1 — Load Data

In [1]:
import pandas as pd
import os

# ── Sesuaikan path ke lokasi file CSV di komputermu ──
MFCC_PATH = 'D:/LISA/.retest_new/MFCC_new.csv'
LPCC_PATH  = 'D:/LISA/.retest_new/LPCC_new.csv'

mfcc_df = pd.read_csv(MFCC_PATH)
lpcc_df = pd.read_csv(LPCC_PATH)

print('MFCC shape :', mfcc_df.shape)
print('LPCC shape :', lpcc_df.shape)

assert 'filename' in mfcc_df.columns, "Kolom 'filename' tidak ditemukan di MFCC!"
assert 'filename' in lpcc_df.columns, "Kolom 'filename' tidak ditemukan di LPCC!"

print('\nContoh filename (5 pertama):')
print(mfcc_df['filename'].head().tolist())
print('\n✅ Cell 1 OK — Data loaded')


MFCC shape : (1050, 2396)
LPCC shape : (1050, 2383)

Contoh filename (5 pertama):
['F04_B1_D0_M2.wav', 'F04_B1_D0_M3.wav', 'F04_B1_D0_M4.wav', 'F04_B1_D0_M5.wav', 'F04_B1_D0_M6.wav']

✅ Cell 1 OK — Data loaded


## ⚙️ Cell 2 — Import & Konfigurasi Global

In [2]:
import numpy as np
import pandas as pd
import os
import time
import gc
from datetime import datetime
import pytz
from threading import Lock

from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from joblib import Parallel, delayed

try:
    from threadpoolctl import threadpool_limits
except Exception:
    threadpool_limits = None

# ── Seed untuk reproduksi hasil ──
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Timezone untuk log waktu WIB ──
WIB = pytz.timezone('Asia/Jakarta')

# ── Jumlah thread paralel untuk evaluasi fitness PSO ──
# Jangan otomatis pakai semua core. Untuk PSO + classifier berat,
# semua core sering membuat laptop/PC throttling dan malah makin lambat.
# Nilai ini biasanya lebih stabil: setengah CPU, dibatasi maksimal 4 worker.
CPU_COUNT = os.cpu_count() or 1
PSO_N_JOBS = max(1, CPU_COUNT - 1)

print(f'CPU count       : {CPU_COUNT}')
print(f'PSO_N_JOBS      : {PSO_N_JOBS}')

# ── Folder output ──
# Sesuaikan path ini jika beda device / beda OS
BASE_DIR = 'D:/LISA/.retest_new/Data_Output_PSO_Scaled'
os.makedirs(BASE_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────
# PARAMETER PSO — 1 set fixed, tidak ada grid search
#
#   N_PARTICLES : jumlah partikel dalam swarm
#                 50 partikel cukup untuk dataset speech — lebih ringan dari
#                 GA karena PSO tidak perlu crossover, hanya update velocity
#   N_ITER      : jumlah iterasi PSO
#                 50 iterasi umumnya cukup untuk konvergen
#   W           : inertia weight — seberapa kuat partikel mempertahankan
#                 arah velocity sebelumnya. Nilai 0.7 = eksplorasi moderat,
#                 partikel tidak terlalu "lengket" di posisi lama
#   C1          : koefisien kognitif — tarikan partikel ke posisi terbaik
#                 dirinya sendiri (pbest). Nilai 1.5 = standar literatur PSO
#   C2          : koefisien sosial — tarikan partikel ke posisi terbaik
#                 seluruh swarm (gbest). Nilai 1.5 = seimbang dengan C1,
#                 artinya partikel sama-sama mempertimbangkan pengalaman
#                 pribadi dan pengetahuan kolektif swarm
#   ALPHA       : koefisien penalti jumlah fitur dalam fungsi fitness
#                 fitness = F1_cv - ALPHA * (n_fitur_dipilih / n_fitur_total)
#                 0.01 = penalti kecil, PSO tetap prioritaskan F1
# ──────────────────────────────────────────────────────
N_PARTICLES = 50
N_ITER      = 50
W           = 0.7
C1          = 1.5
C2          = 1.5
ALPHA       = 0.01

# ──────────────────────────────────────────────────────
# CLASSIFIERS
#
#   SVM         : LinearSVC murni (tanpa kalibrasi). Hanya .predict() yang
#                 dipakai untuk F1, jadi probabilitas tidak diperlukan.
#                 tol=1e-2 agar konvergen cepat pada data ter-scale.
#   RF          : Random Forest, n_jobs=1 karena paralel utama ada di PSO.
#   GaussianNB  : Naive Bayes.
#
# Untuk menonaktifkan salah satu, hapus/comment barisnya.
# Contoh — jalankan RF saja:
#   CLASSIFIERS = { 'RF': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1) }
# ──────────────────────────────────────────────────────
CLASSIFIERS = {
    'SVM': LinearSVC(random_state=RANDOM_STATE, max_iter=5000, tol=1e-2),
    'RF' : RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    'NB' : GaussianNB(),
}

# ── Helper: format detik menjadi "Xm Ys" ──
def format_time(sec):
    m = int(sec // 60)
    s = int(sec % 60)
    return f'{m}m {s}s ({sec:.1f}s)'

# ── Helper: timestamp WIB ──
def get_timestamp_wib():
    return datetime.now(WIB).strftime('%H:%M:%S')

print('\n✅ Cell 2 OK — Config & Classifiers siap')
print(f'   Classifier  : {list(CLASSIFIERS.keys())}')
print(f'   N_PARTICLES={N_PARTICLES} | N_ITER={N_ITER} | W={W} | C1={C1} | C2={C2} | ALPHA={ALPHA}')


CPU count       : 16
PSO_N_JOBS      : 15
Calibration CV  : 3

✅ Cell 2 OK — Config & Classifiers siap
   Classifier  : ['SVM', 'RF', 'NB']
   N_PARTICLES=50 | N_ITER=50 | W=0.7 | C1=1.5 | C2=1.5 | ALPHA=0.01


## 📦 Cell 3 — Persiapan Data

In [3]:
def extract_subject_from_filename(filename):
    """
    Ekstrak label subjek dari nama file.
    Format file: SPEAKER_BATCH_KONDISI_MIC.wav
    Contoh     : CF02_B1_D0_M2.wav  ->  'CF02'

    Catatan:
    - Subjek diambil dari token pertama sebelum underscore.
    - Dengan LOSO, semua sampel dari subjek yang sama akan berada penuh
      di fold validasi atau penuh di fold training.
    """
    basename = os.path.basename(str(filename)).replace('.wav', '')
    parts = basename.split('_')
    if len(parts) < 2:
        raise ValueError(f"Format filename tidak dikenali: '{filename}'")
    return parts[0]


def prepare_features(df):
    """
    Pisahkan DataFrame menjadi:
      X        : array fitur (float32)
      y        : array label integer
      subjects : array label subjek per baris, contoh 'CF02' / 'CM05'

    Kolom non-fitur yang dibuang: Unnamed:0, filename, subject, datatype, label
    """
    subjects  = df['filename'].apply(extract_subject_from_filename).values
    drop_cols = [c for c in ['Unnamed: 0', 'filename', 'subject', 'datatype', 'label'] if c in df.columns]
    X = df.drop(columns=drop_cols).values.astype(np.float32)
    y = df['label'].values
    return X, y, subjects


def prepare_all_datasets(mfcc_df, lpcc_df):
    """
    Bersihkan, split train/test, dan siapkan semua dataset sekaligus.
    Return dict:
      { 'MFCC': (X_tr, X_te, y_tr, y_te, subjects_tr),
        'LPCC': (...),
        'COMBINED': (...) }

    Untuk menonaktifkan salah satu dataset, hapus/comment barisnya di dict datasets.
    """
    for df in [mfcc_df, lpcc_df]:
        df.dropna(inplace=True)
        df.reset_index(drop=True, inplace=True)
        df['label'] = df['label'].astype(int)

    mfcc_tr = mfcc_df[mfcc_df['datatype'] == 'train']
    mfcc_te = mfcc_df[mfcc_df['datatype'] == 'test']
    lpcc_tr = lpcc_df[lpcc_df['datatype'] == 'train']
    lpcc_te = lpcc_df[lpcc_df['datatype'] == 'test']

    X_mfcc_tr, y_tr, subjects_mfcc = prepare_features(mfcc_tr)
    X_mfcc_te, y_te, _              = prepare_features(mfcc_te)
    X_lpcc_tr, _,    subjects_lpcc  = prepare_features(lpcc_tr)
    X_lpcc_te, _,    _              = prepare_features(lpcc_te)

    # ── Opsi A: StandardScaler di-fit SEKALI di train, lalu transform train+test ──
    # Dilakukan di luar loop CV (sekali saja) → biaya hampir nol, tapi LinearSVC/NB
    # jadi ter-scale sehingga konvergen normal (hilang ConvergenceWarning).
    # RF tidak terpengaruh scaling, jadi aman dipakai untuk semua classifier.
    from sklearn.preprocessing import StandardScaler
    _sc_mfcc = StandardScaler().fit(X_mfcc_tr)
    X_mfcc_tr = _sc_mfcc.transform(X_mfcc_tr).astype(np.float32)
    X_mfcc_te = _sc_mfcc.transform(X_mfcc_te).astype(np.float32)
    _sc_lpcc = StandardScaler().fit(X_lpcc_tr)
    X_lpcc_tr = _sc_lpcc.transform(X_lpcc_tr).astype(np.float32)
    X_lpcc_te = _sc_lpcc.transform(X_lpcc_te).astype(np.float32)

    X_comb_tr = np.concatenate([X_mfcc_tr, X_lpcc_tr], axis=1)
    X_comb_te = np.concatenate([X_mfcc_te, X_lpcc_te], axis=1)

    datasets = {
        'MFCC'    : (X_mfcc_tr, X_mfcc_te, y_tr, y_te, subjects_mfcc),
        'LPCC'    : (X_lpcc_tr, X_lpcc_te, y_tr, y_te, subjects_lpcc),
        'COMBINED': (X_comb_tr, X_comb_te, y_tr, y_te, subjects_mfcc),
    }

    unique_s, counts_s = np.unique(subjects_mfcc, return_counts=True)
    print('\n📊 Distribusi subjek di training set (MFCC):')
    for s, c in zip(unique_s, counts_s):
        print(f'   {s}: {c} sampel')
    print(f'   -> Leave-One-Subject-Out membuat {len(unique_s)} fold per eksperimen')

    print('\n📐 Ukuran fitur per dataset:')
    for name, (Xtr, Xte, *_) in datasets.items():
        print(f'   {name:<10}: train={Xtr.shape}, test={Xte.shape}')

    return datasets


datasets = prepare_all_datasets(mfcc_df, lpcc_df)
print('\n✅ Cell 3 OK — Dataset siap')



📊 Distribusi subjek di training set (MFCC):
   F02: 210 sampel
   F05: 210 sampel
   M05: 210 sampel
   M09: 210 sampel
   -> Leave-One-Subject-Out membuat 4 fold per eksperimen

📐 Ukuran fitur per dataset:
   MFCC      : train=(840, 2392), test=(210, 2392)
   LPCC      : train=(840, 2379), test=(210, 2379)
   COMBINED  : train=(840, 4771), test=(210, 4771)

✅ Cell 3 OK — Dataset siap


## 💾 Cell 4 — Fitness Cache

In [4]:
from collections import OrderedDict

MAX_CACHE_SIZE = 5000
fitness_cache = OrderedDict()
_cache_lock = Lock()

def make_cache_key(pos):
    # Pack biner ke byte (8 bit per byte) — ~8x lebih kecil dari .tobytes() langsung
    arr = np.packbits(np.asarray(pos, dtype=np.uint8))
    return arr.tobytes()

def reset_cache():
    global fitness_cache
    with _cache_lock:
        fitness_cache = OrderedDict()
    gc.collect()
    print('Cache direset')

def get_cache(key):
    with _cache_lock:
        value = fitness_cache.get(key)
        if value is not None:
            fitness_cache.move_to_end(key)
        return value

def set_cache(key, value):
    with _cache_lock:
        if key in fitness_cache:
            fitness_cache.move_to_end(key)
            fitness_cache[key] = value
            return
        if len(fitness_cache) >= MAX_CACHE_SIZE:
            fitness_cache.popitem(last=False)
        fitness_cache[key] = value

print('Cell 4 OK — Cache siap (packed key)')


Cell 4 OK — Cache siap (packed key)


## 🧬 Cell 5 — Evaluasi Cross-Subject (LOSO) & Fungsi Fitness


In [5]:
def make_loso_folds(subjects):
    subjects = np.asarray(subjects)
    folds = []
    for s in np.unique(subjects):
        val_idx = np.flatnonzero(subjects == s)
        tr_idx  = np.flatnonzero(subjects != s)
        if len(tr_idx) > 0 and len(val_idx) > 0:
            folds.append((tr_idx, val_idx))
    return folds


def make_loso_fold_arrays(X, y, folds):
    """
    Pre-slice X dan y per fold SEKALI di awal.
    Return list of (X_tr, X_val, y_tr, y_val).
    Hemat alokasi memori berulang di dalam fitness_function.
    """
    result = []
    for tr_idx, val_idx in folds:
        result.append((
            np.ascontiguousarray(X[tr_idx]),
            np.ascontiguousarray(X[val_idx]),
            y[tr_idx],
            y[val_idx],
        ))
    return result


def evaluate_model_presliced(fold_arrays, model, feature_idx):
    """
    Evaluasi LOSO dengan fold yang sudah di-pre-slice.
    Indexing kolom saja (murah), tidak membuat salinan baris lagi.
    """
    scores = []
    for X_tr, X_val, y_tr, y_val in fold_arrays:
        m = clone(model)
        if feature_idx is None:
            m.fit(X_tr, y_tr)
            pred = m.predict(X_val)
        else:
            m.fit(X_tr[:, feature_idx], y_tr)
            pred = m.predict(X_val[:, feature_idx])
        scores.append(f1_score(y_val, pred, average='weighted'))
    if not scores:
        return 0.0, 0.0
    return float(np.mean(scores)), float(np.std(scores))


def fitness_function(pos, fold_arrays, model, alpha, n_feat_total):
    key = make_cache_key(pos)
    cached = get_cache(key)
    if cached is not None:
        return cached[0]

    idx = np.flatnonzero(pos)
    if len(idx) == 0:
        set_cache(key, (0.0, 0.0, 0.0))
        return 0.0

    mean_f1, std_f1 = evaluate_model_presliced(fold_arrays, model, idx)
    fitness = mean_f1 - alpha * (len(idx) / n_feat_total)
    set_cache(key, (fitness, mean_f1, std_f1))
    return fitness


def get_cached_stats(pos):
    cached = get_cache(make_cache_key(pos))
    return cached if cached is not None else (0.0, 0.0, 0.0)


print('Cell 5 OK — Evaluasi LOSO pre-sliced siap')


Cell 5 OK — Evaluasi LOSO pre-sliced siap


## 🔁 Cell 6 — Fungsi Utama PSO

In [6]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -10, 10)))


def run_pso(X, y, subjects, model, n_particles, n_iter, w, c1, c2, alpha,
            n_jobs=None):
    X = np.ascontiguousarray(X)
    y = np.asarray(y)
    n_feat = X.shape[1]

    # n_jobs per-classifier: SVM lebih efisien di worker lebih sedikit
    if n_jobs is None:
        model_type = type(model).__name__
        if model_type == 'LinearSVC':
            n_jobs_eval = min(6, max(1, os.cpu_count() - 1))
        else:
            n_jobs_eval = max(1, os.cpu_count() - 1)
    else:
        n_jobs_eval = n_jobs

    # Pre-compute fold LOSO sekali
    loso_folds = make_loso_folds(subjects)
    if len(loso_folds) < 2:
        raise ValueError('LOSO butuh minimal 2 subjek.')

    # Pre-slice fold arrays sekali — ini penghematan terbesar
    fold_arrays = make_loso_fold_arrays(X, y, loso_folds)

    positions  = np.random.randint(0, 2, (n_particles, n_feat), dtype=np.uint8)
    empty = np.flatnonzero(positions.sum(axis=1) == 0)
    for i in empty:
        positions[i, np.random.randint(n_feat)] = 1

    velocities    = np.random.uniform(-1, 1, (n_particles, n_feat)).astype(np.float32)
    pbest_pos     = positions.copy()
    pbest_fitness = np.full(n_particles, -np.inf, dtype=np.float64)
    gbest_pos     = positions[0].copy()
    gbest_fitness = -np.inf

    start = time.time()

    for it in range(n_iter):
        iter_start = time.time()

        try:
            if threadpool_limits is not None:
                with threadpool_limits(limits=1):
                    fitness_vals = np.array(
                        Parallel(n_jobs=n_jobs_eval, prefer='threads', batch_size='auto')(
                            delayed(fitness_function)(pos, fold_arrays, model, alpha, n_feat)
                            for pos in positions
                        ), dtype=np.float64
                    )
            else:
                fitness_vals = np.array(
                    Parallel(n_jobs=n_jobs_eval, prefer='threads', batch_size='auto')(
                        delayed(fitness_function)(pos, fold_arrays, model, alpha, n_feat)
                        for pos in positions
                    ), dtype=np.float64
                )
        except Exception as e:
            print(f'Parallel gagal iter {it+1}, fallback sequential: {e}')
            fitness_vals = np.array([
                fitness_function(pos, fold_arrays, model, alpha, n_feat)
                for pos in positions
            ], dtype=np.float64)

        improved = fitness_vals > pbest_fitness
        if np.any(improved):
            pbest_fitness[improved] = fitness_vals[improved]
            pbest_pos[improved] = positions[improved].copy()

        best_idx = int(np.argmax(fitness_vals))
        if fitness_vals[best_idx] > gbest_fitness:
            gbest_fitness = float(fitness_vals[best_idx])
            gbest_pos     = positions[best_idx].copy()

        _, f1mean, f1std = get_cached_stats(gbest_pos)
        n_feat_sel   = int(gbest_pos.sum())
        elapsed      = time.time() - start
        iter_time    = time.time() - iter_start
        avg_iter     = elapsed / (it + 1)
        remaining    = avg_iter * (n_iter - it - 1)

        print(
            f'  Iter {it+1:>3}/{n_iter} | '
            f'[{get_timestamp_wib()} WIB] |  ('
            f'iter_best={float(np.max(fitness_vals)):.4f} | '
            f'iter_avg={float(np.mean(fitness_vals)):.4f})  |   '
            f'Fitness={gbest_fitness:.4f} | '
            f'F1cv={f1mean:.4f} | '
            f'Std={f1std:.4f} | '
            f'Feat={n_feat_sel}/{n_feat} | '
            f'Cache={len(fitness_cache)} | '
            f'Time={format_time(iter_time)} | '
            f'Elapsed={format_time(elapsed)} | '
            f'ETA={format_time(remaining)}'
        )

        r1 = np.random.rand(n_particles, n_feat).astype(np.float32)
        r2 = np.random.rand(n_particles, n_feat).astype(np.float32)
        velocities = (
            w  * velocities
            + c1 * r1 * (pbest_pos.astype(np.float32) - positions.astype(np.float32))
            + c2 * r2 * (gbest_pos.astype(np.float32) - positions.astype(np.float32))
        ).astype(np.float32)

        probs     = sigmoid(velocities)
        positions = (np.random.rand(n_particles, n_feat) < probs).astype(np.uint8)
        empty     = np.flatnonzero(positions.sum(axis=1) == 0)
        for i in empty:
            positions[i, np.random.randint(n_feat)] = 1

    _, f1mean, f1std = get_cached_stats(gbest_pos)
    return gbest_pos, gbest_fitness, f1mean, f1std


print('Cell 6 OK — PSO optimized siap')


Cell 6 OK — PSO optimized siap


## 🏃 Cell 7 — Runner Eksperimen (3 Dataset × 3 Classifier)

In [7]:
# ──────────────────────────────────────────────────────
# RUNNER UTAMA + AUTO RESUME / AUTO SKIP
#
# Loop eksperimen: untuk setiap dataset, untuk setiap classifier,
# jalankan PSO -> evaluasi test set -> simpan hasil.
#
# Fitur resume:
#   - Jika pso_results_live.csv sudah ada, hasil lama akan dibaca dulu.
#   - Eksperimen yang exp_id-nya sudah ada dan parameternya sama akan di-skip.
#   - Eksperimen baru / belum selesai akan tetap dijalankan.
#
# Catatan penting:
#   Auto-skip dicek berdasarkan:
#     exp_id + n_particles + n_iter + w + c1 + c2 + alpha
#   Jadi kalau parameter PSO kamu berubah, eksperimen lama tidak otomatis di-skip.
#   datasets_to_run = {k: datasets[k] for k in ['MFCC', 'LPCC', 'COMBINED']}
#   clf_to_run = {k: CLASSIFIERS[k] for k in ['SVM', 'RF', 'NB']}
# ──────────────────────────────────────────────────────

datasets_to_run = {k: datasets[k] for k in [ 'LPCC']}
clf_to_run = {k: CLASSIFIERS[k] for k in ['SVM']}

LIVE_RESULTS_PATH  = os.path.join(BASE_DIR, 'pso_results_live.csv')
FINAL_RESULTS_PATH = os.path.join(BASE_DIR, 'pso_results_final.csv')

# ── Load hasil live lama jika ada ──
if os.path.exists(LIVE_RESULTS_PATH):
    df_live = pd.read_csv(LIVE_RESULTS_PATH)

    # Bersihkan kemungkinan duplikat exp_id, ambil hasil terakhir
    if 'exp_id' in df_live.columns:
        df_live = df_live.drop_duplicates(subset=['exp_id'], keep='last')

    results = df_live.to_dict('records')
    print(f'🔁 Resume aktif: ditemukan {len(results)} hasil lama dari pso_results_live.csv')
else:
    df_live = pd.DataFrame()
    results = []
    print('🆕 Tidak ada pso_results_live.csv lama — mulai eksperimen dari awal')


def is_experiment_completed(df_live, exp_id):
    """
    Return True jika exp_id sudah ada di live CSV dan parameter PSO-nya sama.
    Ini mencegah salah skip ketika kamu mengubah N_PARTICLES / N_ITER / W / C1 / C2 / ALPHA.
    """
    if df_live.empty or 'exp_id' not in df_live.columns:
        return False

    row = df_live[df_live['exp_id'].astype(str) == str(exp_id)]
    if row.empty:
        return False

    row = row.iloc[-1]

    checks = {
        'n_particles': N_PARTICLES,
        'n_iter'     : N_ITER,
        'w'          : W,
        'c1'         : C1,
        'c2'         : C2,
        'alpha'      : ALPHA,
    }

    for col, current_value in checks.items():
        # Kalau CSV lama belum punya kolom parameter, anggap belum aman untuk skip
        if col not in row.index:
            return False

        try:
            if float(row[col]) != float(current_value):
                return False
        except Exception:
            if row[col] != current_value:
                return False

    return True


def save_live_results(results):
    """
    Simpan live CSV dengan aman:
    - Buang duplikat exp_id
    - Ambil hasil terakhir jika ada duplikat
    """
    df_tmp = pd.DataFrame(results)
    if not df_tmp.empty and 'exp_id' in df_tmp.columns:
        df_tmp = df_tmp.drop_duplicates(subset=['exp_id'], keep='last')
    df_tmp.to_csv(LIVE_RESULTS_PATH, index=False)
    return df_tmp


runner_start = time.time()
total    = len(datasets_to_run) * len(clf_to_run)
counter  = 0
skipped  = 0
executed = 0


print('=' * 70)
print('🚀 PSO FEATURE SELECTION — MULAI')
print(f'   Dataset    : {list(datasets_to_run.keys())}')
print(f'   Classifier : {list(clf_to_run.keys())}')
print(f'   Total      : {total} eksperimen')
print(f'   N_PARTICLES={N_PARTICLES} | N_ITER={N_ITER} | W={W} | C1={C1} | C2={C2} | ALPHA={ALPHA}')
print('=' * 70)

for ds_name, (X_tr, X_te, y_tr, y_te, subjects_tr) in datasets_to_run.items():

    # Reset cache setiap ganti dataset — dimensi fitur berbeda antar dataset
    reset_cache()

    for clf_name, clf in clf_to_run.items():
        reset_cache()
        counter += 1
        exp_id  = f'{ds_name}_{clf_name}'

        # ── AUTO SKIP jika eksperimen sudah ada di pso_results_live.csv ──
        if is_experiment_completed(df_live, exp_id):
            skipped += 1
            print(f'\n⏭️  SKIP [{counter}/{total}] {exp_id}')
            print('   Alasan: hasil sudah ada di pso_results_live.csv dan parameter PSO sama')
            continue

        executed += 1

        # ── ETA kasar khusus eksperimen yang benar-benar dijalankan ──
        if executed > 1:
            elapsed_total   = time.time() - runner_start
            avg_exp_time    = elapsed_total / (executed - 1)
            remaining_exps  = total - counter + 1
            eta_global      = avg_exp_time * remaining_exps
            eta_str         = f'ETA global: {format_time(eta_global)}'
        else:
            eta_str = 'ETA global: menghitung...'

        print(f'\n{"=" * 70}')
        print(f'🟢 [{counter}/{total}] {exp_id}  |  {eta_str}')
        print(f'   Dataset   : {ds_name} ({X_tr.shape[1]} fitur, {X_tr.shape[0]} sampel)')
        print(f'   Classifier: {clf_name}')
        print(f'   Mulai     : {get_timestamp_wib()} WIB')
        print(f'{"=" * 70}')

        exp_start = time.time()

        # ── Jalankan PSO ──
        best_pos, best_fit, f1_cv_mean, f1_cv_std = run_pso(
            X_tr, y_tr, subjects_tr, clf,
            N_PARTICLES, N_ITER, W, C1, C2, ALPHA
        )

        # ── Evaluasi pada test set menggunakan fitur hasil seleksi PSO ──
        selected_idx = np.where(best_pos)[0]
        n_selected   = len(selected_idx)

        model_test = clone(clf)
        model_test.fit(X_tr[:, selected_idx], y_tr)
        pred_te = model_test.predict(X_te[:, selected_idx])
        f1_test = float(f1_score(y_te, pred_te, average='weighted'))

        exp_time = time.time() - exp_start

        print(f'\n✅ SELESAI [{counter}/{total}] {exp_id}')
        print(f'   Selesai       : {get_timestamp_wib()} WIB')
        print(f'   Durasi        : {format_time(exp_time)}')
        print(f'   F1 Test       : {f1_test:.4f}')
        print(f'   F1 CV (LOSO)  : {f1_cv_mean:.4f} ± {f1_cv_std:.4f}')
        print(f'   Gap CV<->Test : {abs(f1_cv_mean - f1_test):.4f}')
        print(f'   Fitur terpilih: {n_selected}/{X_tr.shape[1]} ({100*n_selected/X_tr.shape[1]:.1f}%)')

        results.append({
            'exp_id'          : exp_id,
            'dataset'         : ds_name,
            'classifier'      : clf_name,
            'f1_test'         : round(f1_test, 4),
            'f1_cv_mean'      : round(f1_cv_mean, 4),
            'f1_cv_std'       : round(f1_cv_std, 4),
            'f1_gap'          : round(abs(f1_cv_mean - f1_test), 4),
            'best_fitness'    : round(best_fit, 4),
            'n_feat_selected' : n_selected,
            'n_feat_total'    : X_tr.shape[1],
            'feat_ratio_pct'  : round(100 * n_selected / X_tr.shape[1], 2),
            'duration_s'      : round(exp_time, 2),
            'n_particles'     : N_PARTICLES,
            'n_iter'          : N_ITER,
            'w'               : W,
            'c1'              : C1,
            'c2'              : C2,
            'alpha'           : ALPHA,
        })

        # Simpan hasil sementara setiap eksperimen selesai
        df_live = save_live_results(results)

# ──────────────────────────────────────────────────────
# HASIL AKHIR
# ──────────────────────────────────────────────────────
df_results = pd.DataFrame(results)

if df_results.empty:
    print('\n⚠️ Tidak ada hasil eksperimen untuk ditampilkan.')
else:
    if 'exp_id' in df_results.columns:
        df_results = df_results.drop_duplicates(subset=['exp_id'], keep='last')

    df_results = (
        df_results
        .sort_values('f1_test', ascending=False)
        .reset_index(drop=True)
    )

    df_results.to_csv(FINAL_RESULTS_PATH, index=False)

    total_time = time.time() - runner_start
    print(f'\n{"=" * 70}')
    print(f'🏁 SEMUA EKSPERIMEN SELESAI — {format_time(total_time)}')
    print(f'   Dijalankan baru : {executed}')
    print(f'   Di-skip         : {skipped}')
    print(f'   Total hasil     : {len(df_results)}')
    print(f'{"=" * 70}')

    header = f"{'No':>3} | {'Eksperimen':<20} | {'F1 Test':>7} | {'F1 CV':>6} | {'Std':>6} | {'Gap':>6} | {'Fitur':<18} | Durasi"
    print(header)
    print('-' * len(header))
    for i, row in df_results.iterrows():
        feat_str = f"{int(row['n_feat_selected'])}/{int(row['n_feat_total'])} ({row['feat_ratio_pct']:.1f}%)"
        print(
            f"{i+1:>3} | {row['exp_id']:<20} | "
            f"{row['f1_test']:>7.4f} | {row['f1_cv_mean']:>6.4f} | {row['f1_cv_std']:>6.4f} | {row['f1_gap']:>6.4f} | "
            f"{feat_str:<18} | {format_time(row['duration_s'])}"
        )
    print('-' * len(header))

    display(df_results)


🔁 Resume aktif: ditemukan 1 hasil lama dari pso_results_live.csv
🚀 PSO FEATURE SELECTION — MULAI
   Dataset    : ['LPCC']
   Classifier : ['SVM']
   Total      : 1 eksperimen
   N_PARTICLES=50 | N_ITER=50 | W=0.7 | C1=1.5 | C2=1.5 | ALPHA=0.01
Cache direset
Cache direset

⏭️  SKIP [1/1] LPCC_SVM
   Alasan: hasil sudah ada di pso_results_live.csv dan parameter PSO sama

🏁 SEMUA EKSPERIMEN SELESAI — 0m 0s (0.1s)
   Dijalankan baru : 0
   Di-skip         : 1
   Total hasil     : 1
 No | Eksperimen           | F1 Test |  F1 CV |    Std |    Gap | Fitur              | Durasi
---------------------------------------------------------------------------------------------
  1 | LPCC_SVM             |  0.1015 | 0.1904 | 0.0573 | 0.0889 | 1186/2379 (49.9%)  | 81m 52s (4912.2s)
---------------------------------------------------------------------------------------------


,exp_id,dataset,classifier,f1_test,f1_cv_mean,f1_cv_std,f1_gap,best_fitness,n_feat_selected,n_feat_total,feat_ratio_pct,duration_s,n_particles,n_iter,w,c1,c2,alpha
0,LPCC_SVM,LPCC,SVM,0.1015,0.1904,0.0573,0.0889,0.1854,1186,2379,49.85,4912.16,50,50,0.7,1.5,1.5,0.01


## 📊 Cell 8 — Ringkasan Per Dataset & Per Classifier

In [8]:
# Rata-rata F1 Test per dataset
print('📈 Rata-rata F1 Test per Dataset:')
for ds, grp in df_results.groupby('dataset'):
    print(f'   {ds:<10}: {grp["f1_test"].mean():.4f} ± {grp["f1_test"].std():.4f}')

print()

# Rata-rata F1 Test per classifier
print('🤖 Rata-rata F1 Test per Classifier:')
for clf, grp in df_results.groupby('classifier'):
    print(f'   {clf:<10}: {grp["f1_test"].mean():.4f} ± {grp["f1_test"].std():.4f}')

print()

# Eksperimen terbaik
best = df_results.iloc[0]
print(f'🏆 Terbaik : {best["exp_id"]}')
print(f'   F1 Test  : {best["f1_test"]:.4f}')
print(f'   F1 CV    : {best["f1_cv_mean"]:.4f} ± {best["f1_cv_std"]:.4f}')
print(f'   Gap      : {best["f1_gap"]:.4f}')
print(f'   Fitur    : {int(best["n_feat_selected"])}/{int(best["n_feat_total"])} ({best["feat_ratio_pct"]:.1f}%)')


📈 Rata-rata F1 Test per Dataset:
   LPCC      : 0.1015 ± nan

🤖 Rata-rata F1 Test per Classifier:
   SVM       : 0.1015 ± nan

🏆 Terbaik : LPCC_SVM
   F1 Test  : 0.1015
   F1 CV    : 0.1904 ± 0.0573
   Gap      : 0.0889
   Fitur    : 1186/2379 (49.9%)
